# Re-verify this proof

This notebook re-runs the exact `proof.py` deposited to Zenodo and prints a verdict.

1. **Run All Cells** — Run menu → Run All Cells (or Shift+Enter through each).
2. The proof fetches from Zenodo (preloaded from the link you clicked), executes, and prints a verdict at the bottom.
3. Expect 10–60 seconds total.

**To inspect the source first:** the proof's page on https://yaniv-golan.github.io/proof-engine/ shows `proof.py` inline, syntax-highlighted, in the "View proof source" section.

**To verify a different proof:** edit the `DOI` variable in cell 1 and re-run all cells.

<sub>Every cell below is inspectable. There is no hidden orchestration — what you see is what runs.</sub>

In [ ]:
# Which proof do you want to verify?
#
# Leave DOI_OVERRIDE = None to use the DOI preloaded from the ?doi= query string
# (written to /tmp/binder_doi by the binder_doi_capture server extension — this
# is the normal path when you arrived from a proof page).
#
# To verify a different proof, set DOI_OVERRIDE to its Zenodo DOI, then re-run
# all cells. The override wins unconditionally.
DOI_OVERRIDE = None
# DOI_OVERRIDE = "10.5281/zenodo.19635623"  # example: eml-calculator-closure

import os, re
DOI_RE = re.compile(r"^10\.\d{4,9}/zenodo\.\d+$")
DOI_PRELOAD_FILE = "/tmp/binder_doi"

if DOI_OVERRIDE:
    DOI = DOI_OVERRIDE.strip()
    DOI_SOURCE = "manual override"
elif os.path.exists(DOI_PRELOAD_FILE):
    DOI = open(DOI_PRELOAD_FILE).read().strip()
    DOI_SOURCE = "preloaded from ?doi= query string"
else:
    DOI = "10.5281/zenodo.19455603"  # fallback example proof
    DOI_SOURCE = "built-in example fallback"

assert DOI_RE.fullmatch(DOI), f"Invalid Zenodo DOI: {DOI!r}. Expected 10.5281/zenodo.XXXXXXX."
zenodo_id = DOI.split("/zenodo.")[-1]
print(f"DOI:       {DOI}  ({DOI_SOURCE})")
print(f"Zenodo ID: {zenodo_id}")

In [ ]:
# Fetch proof.py from Zenodo. Zenodo deposits are immutable, so re-running
# this in a year fetches the same bytes. The "View proof source" section
# on the proof's page shows what you're about to execute.
import requests

meta = requests.get(f"https://zenodo.org/api/records/{zenodo_id}", timeout=30).json()
if "files" not in meta:
    raise RuntimeError(f"Zenodo record {zenodo_id} has no files. Response: {meta!r}")

proof_py_url = next((f["links"]["self"] for f in meta["files"] if f["key"] == "proof.py"), None)
if not proof_py_url:
    raise FileNotFoundError(f"No proof.py in Zenodo record {zenodo_id}")

proof_text = requests.get(proof_py_url, timeout=60).text
print(f"Fetched:  {proof_py_url}")
print(f"Size:     {len(proof_text):,} bytes, {proof_text.count(chr(10)):,} lines")

In [ ]:
# Run the proof. Output streams live to this cell AND is captured to a
# buffer so the next cell can extract the verdict from a single execution.
# The synthetic __file__ anchors legacy proofs' 4-dirname traversal onto
# the ~/proof-engine clone that postBuild created.
import io, sys
from IPython.display import Markdown, display

repo_root = os.path.expanduser("~/proof-engine")
proof_engine_root = os.path.join(repo_root, "proof-engine", "skills", "proof-engine")
if not os.path.isdir(proof_engine_root):
    raise RuntimeError(f"proof-engine clone missing at {proof_engine_root} — image may be broken.")
os.environ["PROOF_ENGINE_ROOT"] = proof_engine_root
if proof_engine_root not in sys.path:
    sys.path.insert(0, proof_engine_root)

proof_script_name = f"proof_{zenodo_id}.py"
synthetic_path = os.path.join(repo_root, "site", "proofs", "_runtime", proof_script_name)

class _Tee:
    def __init__(self, *streams): self.streams = streams
    def write(self, s):
        for st in self.streams: st.write(s)
        return len(s)
    def flush(self):
        for st in self.streams:
            if hasattr(st, "flush"): st.flush()

proof_output = io.StringIO()
display(Markdown("### Proof output"))
_orig_stdout = sys.stdout
sys.stdout = _Tee(_orig_stdout, proof_output)
try:
    exec(compile(proof_text, proof_script_name, "exec"),
         {"__name__": "__main__", "__file__": synthetic_path})
finally:
    sys.stdout = _orig_stdout

captured = proof_output.getvalue()
print(f"\n--- captured {len(captured):,} bytes for verdict extraction ---")

In [ ]:
# Extract the verdict. Primary path: parse the canonical JSON summary block
# emitted after `=== PROOF SUMMARY (JSON) ===` (same marker used by
# tools/lib/proof_runner.py — every new proof emits this, and it's the
# machine-readable source of truth). Fallback path: regex across multiple
# legacy verdict print formats for older proofs that predate the JSON
# summary (a handful of proofs still print only `VERDICT: X` or
# `=== VERDICT: X ===` without the JSON block).
#
# Also resolve this DOI to a proof-page URL via the site's doi-index.json,
# so the "View proof page" link points to the *specific* proof (with its
# "View proof source" section), not to the landing page.
import json
from IPython.display import Markdown, display

verdict = "UNKNOWN"
unverified = False

JSON_MARKER = "=== PROOF SUMMARY (JSON) ==="
_idx = captured.find(JSON_MARKER)
if _idx != -1:
    try:
        _json_tail = captured[_idx + len(JSON_MARKER):].strip()
        # The JSON object may be followed by other output; take everything
        # up to and including the matching closing brace.
        _depth = 0
        _end = None
        for _i, _ch in enumerate(_json_tail):
            if _ch == "{":
                _depth += 1
            elif _ch == "}":
                _depth -= 1
                if _depth == 0:
                    _end = _i + 1
                    break
        if _end is not None:
            _data = json.loads(_json_tail[:_end])
            _v = (_data.get("verdict") or "").strip()
            if _v:
                # Strip the "(with unverified citations)" suffix if present.
                if "(with unverified citations)" in _v:
                    unverified = True
                    _v = _v.replace("(with unverified citations)", "").strip()
                verdict = _v
    except (json.JSONDecodeError, ValueError):
        pass  # fall through to regex

if verdict == "UNKNOWN":
    # Fallback: regex across legacy print formats. Covers:
    #   "Verdict: X", "VERDICT: X", "  VERDICT: X", "=== VERDICT: X ==="
    # (Plain "=== VERDICT ===" header with no inline value is skipped here;
    # those proofs always also print "VERDICT: X" on a subsequent line.)
    VERDICT_RE = re.compile(
        r"(?:^|\s)(?:===\s*)?(?:VERDICT|Verdict)\s*:?\s*"
        r"([A-Z][A-Z ]+?)(?:\s*\(with unverified citations\))?"
        r"(?:\s*===)?\s*$",
        re.MULTILINE,
    )
    m = VERDICT_RE.search(captured)
    if m:
        verdict = m.group(1).strip()
        if "with unverified citations" in captured:
            unverified = True

emoji = {
    "PROVED": "✅", "DISPROVED": "❌", "SUPPORTED": "🟢",
    "PARTIALLY VERIFIED": "🟡", "UNDETERMINED": "⚪",
}.get(verdict, "❔")
suffix = " *(with unverified citations)*" if unverified else ""

# Resolve DOI → proof page URL via the site's doi-index.json.
# Graceful fallback if the lookup fails (offline, hash mismatch, etc.):
# fall back to the landing page — still useful, just less precise.
SITE_BASE = "https://yaniv-golan.github.io/proof-engine"
proof_page_url = SITE_BASE + "/"
try:
    idx = requests.get(f"{SITE_BASE}/doi-index.json", timeout=10).json()
    slug = idx.get(DOI.lower())
    if slug:
        proof_page_url = f"{SITE_BASE}/proofs/{slug}/"
except Exception as _e:
    pass  # keep fallback URL

display(Markdown(f"## {emoji} Verdict: **{verdict}**{suffix}"))
display(Markdown(
    f"[View proof page]({proof_page_url}#proof-source) · "
    f"[View on Zenodo](https://doi.org/{DOI})"
))

---

*Launcher repo: [yaniv-golan/proof-engine-binder](https://github.com/yaniv-golan/proof-engine-binder). See [yaniv-golan/proof-engine](https://github.com/yaniv-golan/proof-engine) for the proofs and skill.*